<a href="https://colab.research.google.com/github/GokulM8/Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Setup — connect to the warehouse (DuckDB, remote)

Same lane as ML-04/ML-07: **Content Refresh / Opportunity Scoring**, same table (`fact_content_daily_performance`). This week the label changes: instead of the within-month proxy used for the baseline, this pulls a real forward label from April 2026 (still not the sealed June month).

In [1]:
%pip install -q duckdb huggingface_hub

In [3]:
from google.colab import userdata
from huggingface_hub import HfApi
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
import duckdb
import numpy as np
import pandas as pd

# Token comes from the Colab Secret, never printed — this repo is public.
HF_TOKEN = userdata.get("HF_TOKEN")

In [4]:
api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
config_files = sorted(f for f in all_files if "fact_content_daily_performance" in f and f.endswith(".parquet"))
print(f"Found {len(config_files)} parquet file(s) for this config.")

Found 19 parquet file(s) for this config.


In [5]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

remote_paths = [f"hf://datasets/FlyRank/internship-warehouse/{f}" for f in config_files]
paths_sql = "[" + ", ".join(f"'{p}'" for p in remote_paths) + "]"
con.sql(f"CREATE OR REPLACE VIEW fact AS SELECT * FROM read_parquet({paths_sql})")

In [6]:
# March features — predictors only, all knowable at month-close (same 5 features as ML-04/ML-07,
# plus the raw click halves so the baseline rule can still be recomputed for comparison)
march_features = con.sql("""
    WITH march AS (
        SELECT *, CAST(strftime(report_date, '%d') AS INTEGER) AS day,
            sessions_organic + sessions_direct + sessions_referral
            + sessions_social + sessions_paid + sessions_ai AS total_sessions_row
        FROM fact
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    )
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS total_impressions,
        SUM(sessions_ai) * 1.0 / NULLIF(SUM(total_sessions_row), 0) AS ai_search_share,
        SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,
        SUM(gsc_clicks) AS march_total_clicks,
        SUM(CASE WHEN day <= 15 THEN gsc_clicks ELSE 0 END) AS front_half_clicks,
        SUM(CASE WHEN day > 15 THEN gsc_clicks ELSE 0 END) AS back_half_clicks
    FROM march
    GROUP BY client_hash_id, content_hash_id
""").df().fillna(0)

print("March rows:", len(march_features))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 331437


In [7]:
# April totals — used ONLY to build the forward label, never as a model input
april_totals = con.sql("""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_total_clicks
    FROM fact
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
    GROUP BY client_hash_id, content_hash_id
""").df()

print("April rows:", len(april_totals))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April rows: 362172


In [8]:
# Join March predictors to the April outcome (inner join — a page needs both months to get a label)
joined = march_features.merge(april_totals, on=["client_hash_id", "content_hash_id"], how="inner")

joined["is_declining_forward"] = (
    joined["april_total_clicks"] < joined["march_total_clicks"]
).astype(int)

print("Rows with both months present:", len(joined))
print(joined["is_declining_forward"].value_counts(normalize=True))

Rows with both months present: 331436
is_declining_forward
0    0.863919
1    0.136081
Name: proportion, dtype: float64


### Method choice, in plain words

**Target:** `is_declining_forward` — did this page's clicks actually go down from March to April? This is a real forward label, distinct from the Week-4 baseline's within-month proxy (front vs. back half of the same month) — that proxy was a reasonable guess at decline, but never checked against what actually happened next.

**Method: Random Forest Classifier.** Clustering doesn't fit — there's a real label to predict, not an unsupervised grouping question. A plain correlation/signal read (like the Week-4 signal checks) can't combine multiple weak signals into one decision. Logistic Regression was the other real candidate, but a page's risk plausibly depends on *combinations* — e.g. good position but collapsing CTR is a different risk profile than bad position with stable CTR — which a linear model can blur together. Gradient Boosting was considered but skipped for now: after the inner join to April, the usable row count is modest, and boosting overfits small, noisy tabular data faster than a shallow, regularized Random Forest does. Random Forest also plugs cleanly into permutation importance for the error analysis in section 4.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, not random row-shuffled.** Pages from the same client tend to share a lot — the same site template, the same seasonal calendar, the same one-off event that may have hit all their pages in April. A random split could put some of a client's pages in train and others in test, letting the model learn "this smells like Client X" rather than a genuine content-risk pattern, and quietly inflate the score. `GroupShuffleSplit` on `client_hash_id` keeps every client entirely on one side.

In [9]:
feature_cols = ["avg_ctr", "avg_position", "total_impressions", "ai_search_share", "engagement_rate"]

X = joined[feature_cols].fillna(0)
y = joined["is_declining_forward"]
groups = joined["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

train_df = joined.iloc[train_idx].reset_index(drop=True)
test_df = joined.iloc[test_idx].reset_index(drop=True)

print("Train rows:", len(train_df), " Test rows:", len(test_df))
print("Clients in train:", train_df["client_hash_id"].nunique(), " Clients in test:", test_df["client_hash_id"].nunique())
print("Client overlap between train/test:", set(train_df["client_hash_id"]) & set(test_df["client_hash_id"]))

Train rows: 281613  Test rows: 49823
Clients in train: 38  Clients in test: 17
Client overlap between train/test: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
# Recompute the Week-4 baseline rule on the SAME test split — no retraining needed, it's a fixed rule
test_df = test_df.copy()
test_df["decline_magnitude"] = (test_df["front_half_clicks"] - test_df["back_half_clicks"]).clip(lower=0)
test_df["baseline_score"] = test_df["decline_magnitude"] * np.log1p(test_df["total_impressions"])

In [11]:
X_train = train_df[feature_cols].fillna(0)
y_train = train_df["is_declining_forward"]
X_test = test_df[feature_cols].fillna(0)
y_test = test_df["is_declining_forward"]

model = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_leaf=10,
    class_weight="balanced", random_state=42,
)
model.fit(X_train, y_train)
test_df["model_proba"] = model.predict_proba(X_test)[:, 1]

In [12]:
def precision_at_k(df, score_col, label_col, k=10):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

comparison = pd.DataFrame({
    "system": ["Baseline (Week-4 rule)", "Model (Random Forest)"],
    "roc_auc": [
        roc_auc_score(y_test, test_df["baseline_score"]),
        roc_auc_score(y_test, test_df["model_proba"]),
    ],
    "precision_at_10": [
        precision_at_k(test_df, "baseline_score", "is_declining_forward", 10),
        precision_at_k(test_df, "model_proba", "is_declining_forward", 10),
    ],
})
comparison

,system,roc_auc,precision_at_10
0,Baseline (Week-4 rule),0.686792,0.7
1,Model (Random Forest),0.953919,1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [13]:
perm = permutation_importance(
    model, X_test, y_test, n_repeats=20, random_state=42, scoring="roc_auc"
)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df

,feature,importance_mean,importance_std
0,avg_ctr,0.193174,0.001261
1,avg_position,0.001686,0.000461
4,engagement_rate,0.000026,0.000108
3,ai_search_share,-0.000067,0.000087
2,total_impressions,-0.001187,0.000466


In [14]:
test_df["predicted_label"] = (test_df["model_proba"] >= 0.5).astype(int)

false_positives = test_df[(test_df["predicted_label"] == 1) & (test_df["is_declining_forward"] == 0)]
false_negatives = test_df[(test_df["predicted_label"] == 0) & (test_df["is_declining_forward"] == 1)]

print(f"False positives: {len(false_positives)} of {len(test_df)} test rows")
print(f"False negatives: {len(false_negatives)} of {len(test_df)} test rows")

false_positives[["content_hash_id", "avg_ctr", "avg_position", "total_impressions", "model_proba"]].head(5)

False positives: 4314 of 49823 test rows
False negatives: 0 of 49823 test rows


,content_hash_id,avg_ctr,avg_position,total_impressions,model_proba
326,content_aba6da0198f4e0c9,0.002703,13.479091,370.0,0.916145
398,content_f1b73318a7674dfc,0.004184,7.956035,239.0,0.911276
407,content_f4a0e5c90b283626,0.000240,17.199824,8322.0,0.880323
448,content_38479efc65625f3f,0.002023,6.109699,2472.0,0.922273
520,content_ded5dd5859f8be00,0.000683,3.831293,1465.0,0.898159


In [15]:
false_negatives[["content_hash_id", "avg_ctr", "avg_position", "total_impressions", "model_proba"]].head(5)

,content_hash_id,avg_ctr,avg_position,total_impressions,model_proba


**Where the model is wrong — fill this in after you've looked at the rows above.** A plausible read to check against your real output: false positives may cluster on pages whose March CTR or position looked shaky but that recovered in April for reasons no March-only feature can see (a seasonal keyword, a competitor dropping off, a link picked up elsewhere). False negatives may show pages with strong, stable March numbers that fell off a cliff in April — a sudden shift with no early warning sign in March. Check the permutation importance table too: if `avg_position` or `avg_ctr` dominates, that lines up with (or contradicts) the Week-4 signal verdicts — worth a sentence either way.

## Self-check

Before you submit, confirm each line honestly:

- [-] Every section above is filled — markdown thinking AND the code that backs it
- [-] The notebook runs top to bottom with no errors (Runtime → Run all)
- [-] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [-] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.